# 半导体工业生产指数：ALFRED 旧版本覆盖预检

核对 IPG3344S 与已停用 IPG33441A1S 的公开原始下载表单。工业生产量不同于已关闭的半导体PPI价格机制。先查2011–2022 Stage1训练期是否存在同期版本；本卡不POST下载数值、不读行情、不拟合。

In [1]:
import json,hashlib
from pathlib import Path
from bs4 import BeautifulSoup
from src.data.minute_intake import atomic_json
root=Path.cwd();card=root/'research/semiconductor_ip_alfred_preflight_20261022'
plan=json.loads((card/'plan.json').read_text())
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
archive=json.loads((card/'nas_source_archive_receipt.json').read_text())
assert archive['scoped_restoration_verified'] and archive['remote_sha256_readback']==archive['archive_sha256']
series=[]
for entry in (plan['candidate_main'],plan['historical_discontinued_comparator']):
 path=root/entry['raw_form'];assert sha(path)==entry['raw_form_sha256'] and path.stat().st_size==entry['raw_form_bytes']
 assert archive['source_objects'][entry['raw_form']]['sha256']==entry['raw_form_sha256']
 page=BeautifulSoup(path.read_bytes(),'html.parser')
 form=next(x for x in page.find_all('form') if x.find(attrs={'name':'form[file_type]'}) is not None)
 vintages=[x.get('value') for x in form.find(attrs={'name':'form[selected_vintage_dates][]'}).find_all('option')]
 assert vintages and vintages==sorted(vintages)
 series.append({'series_id':entry['series_id'],'form_sha256':entry['raw_form_sha256'],
   'selectable_vintage_dates':len(vintages),'first_selectable_vintage':vintages[0],
   'last_selectable_vintage':vintages[-1],
   'pre_2015_vintage_dates':sum(date<'2015-01-01' for date in vintages),
   'observation_end_setting':form.find(attrs={'name':'form[obs_end_date]'}).get('value')})
assert series[0]['first_selectable_vintage']=='2015-02-18' and series[1]['first_selectable_vintage']=='2015-02-18'
result={'status':'needs_pre2015_original_vintages_no_numeric_download_no_fit',
 'plan_sha256':sha(card/'plan.json'),'series':series,
 'Stage1_training_first':'2011-01-03','Stage1_training_last':'2022-12-30',
 'vintage_gap_first':'2011-01-03','vintage_gap_last':'2015-02-17',
 'discontinued_series_definition_not_spliced':True,
 'archive_POSTs':0,'market_outcome_values_loaded':0,'new_model_fits':0,'new_native_accounts':0,
 'nas_archive_sha256':archive['archive_sha256'],'nas_scoped_restoration_verified':True,
 'next_action':'Only an original 2011-2014 G.17 release archive plus a separately justified old/new industry-definition bridge could reopen this mechanism. Do not use later revised observations as early vintages or shorten Stage1 training.'}
atomic_json(result,card/'source_preflight_result.json')
result


{'status': 'needs_pre2015_original_vintages_no_numeric_download_no_fit',
 'plan_sha256': '792417e21025d22d5b3dd38dc816558d5cf78ed183e8afb8c8cbf62ac791cd86',
 'series': [{'series_id': 'IPG3344S',
   'form_sha256': 'f8e66db823c627809f153e4876a52c1486b1afceae424802583e503f0556255f',
   'selectable_vintage_dates': 150,
   'first_selectable_vintage': '2015-02-18',
   'last_selectable_vintage': '2026-10-16',
   'pre_2015_vintage_dates': 0,
   'observation_end_setting': '2026-09-01'},
  {'series_id': 'IPG33441A1S',
   'form_sha256': '762fe09d2d7645355c75cb3c7f5918f64fe35df9530dc32650371ec63b73c631',
   'selectable_vintage_dates': 6,
   'first_selectable_vintage': '2015-02-18',
   'last_selectable_vintage': '2015-07-15',
   'pre_2015_vintage_dates': 0,
   'observation_end_setting': '2015-06-01'}],
 'Stage1_training_first': '2011-01-03',
 'Stage1_training_last': '2022-12-30',
 'vintage_gap_first': '2011-01-03',
 'vintage_gap_last': '2015-02-17',
 'discontinued_series_definition_not_spliced': Tr

## 中文结论

两个系列的可选日期版本都从2015-02-18才开始，缺原 Stage1 2011–2014 训练期的同期数据。旧系列在2015年停用，与新系列定义不同，不能直接拼接。原始表单已存NAS并冷恢复。0数值下载、0市场结果、0拟合、0账户；来源路线标记 needs_pre2015_original_vintages。